#1. Gọi thư viện

In [ ]:
import pandas as pd
import numpy as np
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import matplotlib.pyplot as plt
folder = Path("/content/drive/MyDrive/Tân Cảng Hiệp Phước/EDA")


#2. Tỉ lệ mặc hàng rỗng không định danh

In [ ]:
# đọc file
file_path = folder / "data_4.xlsx"
df=pd.read_excel(file_path)
print("in 5 dòng đầu\n", df)

In [ ]:
# Xem ô NaN, rỗng hoặc chỉ chứa khoảng trắng là giá trị thiếu
df_check = df.replace(r'^\s*$', pd.NA, regex=True)

na_stats = pd.DataFrame({
    'Tổng số dòng': len(df),
    'Số lượng NA': df_check.isna().sum(),
    'Số lượng không NA': df_check.notna().sum(),
    'Tỷ lệ NA (%)': (df_check.isna().mean() * 100).round(2)
})

na_stats = na_stats.sort_values('Tỷ lệ NA (%)', ascending=False)

display(na_stats)

#3. Tỉ lệ mặt hàng chủ lực qua các năm

In [ ]:
# Danh sách mặt hàng chủ lực
mat_hang_chu_luc = [
    'LƯỚI AN TOÀN', 'GẠCH', 'BÁCH HÓA', 'BỘT CÁ',
    'PHÂN BÓN', 'MÁY MÓC', 'GẠO', 'MAY MẶC',
    'LẠNH', 'XƠ DỪA', 'MỤN DỪA'
]

# Lọc dữ liệu mặt hàng chủ lực
df_plot = df[
    df['Loại hàng'].isin(mat_hang_chu_luc) &
    df['Năm'].isin([2021, 2022, 2023, 2024, 2025])
].copy()

# Tổng sản lượng TEU theo Năm và Loại hàng
data_plot = (
    df_plot.groupby(['Năm', 'Loại hàng'])['TEU']
    .sum()
    .reset_index()
)

# Pivot: Hàng = Năm, Cột = Loại hàng
data_pivot = data_plot.pivot(
    index='Năm',
    columns='Loại hàng',
    values='TEU'
)

# Sắp xếp theo thứ tự mặt hàng
data_pivot = data_pivot.reindex(columns=mat_hang_chu_luc)

# Vẽ biểu đồ
plt.figure(figsize=(15, 7))

for loai_hang in data_pivot.columns:
    plt.plot(
        data_pivot.index,
        data_pivot[loai_hang],
        marker='o',
        linewidth=2,
        label=loai_hang
    )

plt.xlabel('Năm')
plt.ylabel('Sản lượng TEU')
plt.title('Sản lượng TEU của các mặt hàng chủ lực giai đoạn 2021–2025')

plt.xticks([2021, 2022, 2023, 2024, 2025])
plt.legend(title='Loại hàng', bbox_to_anchor=(1.02, 1), loc='upper left')

plt.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
# Groupby theo Năm và Loại hàng
df_co_cau = (
    df.groupby(['Năm', 'Loại hàng'])['TEU']
    .sum()
    .reset_index()
    .sort_values(['Năm', 'TEU'], ascending=[True, False])
)

# In kết quả
display(df_co_cau)
pivot_co_cau = df_co_cau.pivot(
    index='Loại hàng',
    columns='Năm',
    values='TEU'
).fillna(0)



In [ ]:
# Sắp xếp giảm dần theo tổng TEU của tất cả các năm
pivot_co_cau['Tổng TEU'] = pivot_co_cau.sum(axis=1)

pivot_co_cau = pivot_co_cau.sort_values(
    'Tổng TEU',
    ascending=False
)

display(pivot_co_cau)

In [ ]:
# Lọc dữ liệu giai đoạn 2021–2025
df_plot = df[df['Năm'].isin([2021, 2022, 2023, 2024, 2025])].copy()

# Tính tổng TEU theo từng năm và tháng
monthly = (
    df_plot.groupby(['Năm', 'Tháng'])['TEU']
    .sum()
    .reset_index()
)

# Chuyển dữ liệu thành dạng bảng:
# Hàng = Tháng, Cột = Năm, Giá trị = Tổng TEU
pivot = monthly.pivot(
    index='Tháng',
    columns='Năm',
    values='TEU'
)

# Vẽ biểu đồ 5 đường line
plt.figure(figsize=(13, 6))

for year in [2021, 2022, 2023, 2024, 2025]:
    if year in pivot.columns:
        plt.plot(
            pivot.index,
            pivot[year],
            marker='o',
            linewidth=2,
            label=str(year)
        )

plt.xticks(range(1, 13))
plt.xlabel('Tháng')
plt.ylabel('Tổng sản lượng TEU')
plt.title('So sánh sản lượng TEU theo tháng giai đoạn 2021–2025')

plt.legend(title='Năm')
plt.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

#4. Tỉ lệ mặt hàng các năm theo tháng

In [ ]:
# Lọc các dòng thiếu dữ liệu ở cột Loại hàng
mask_na = (
    df_check['Loại hàng'].isna()
    | df_check['Loại hàng'].astype('string').str.strip().eq('')
)

df_na_loai_hang = df_check[mask_na]

# In số lượng và 5 dòng đầu
print("Số dòng thiếu Loại hàng:", len(df_na_loai_hang))
display(df_na_loai_hang.head())

# Tỷ lệ thiếu (%)
print(f"Tỷ lệ thiếu: {mask_na.mean() * 100:.2f}%")
df_check = df_check[~mask_na].copy()

In [ ]:
# 1. Danh sách mặt hàng và năm
mat_hang_giu = ['XƠ DỪA', 'MỤN DỪA', 'LẠNH', 'MAY MẶC', 'GẠO']
years = [2021, 2022, 2023, 2024, 2025]

# 2. Lọc dữ liệu
df_check = df[
    df['Loại hàng'].isin(mat_hang_giu) &
    df['Năm'].isin(years)
].copy()

print(df_check['Loại hàng'].value_counts())
print("Số dòng sau khi lọc:", len(df_check))

# 3. Tổng sản lượng TEU theo năm, tháng và loại hàng
data_plot = (
    df_check.groupby(['Năm', 'Tháng', 'Loại hàng'])['TEU']
    .sum()
    .reset_index()
)

# 4. Quy định màu sắc thống nhất cho từng mặt hàng
colors = {
    'XƠ DỪA': '#9467BD',
    'MỤN DỪA': '#D62728',
    'LẠNH': '#FF7F0E',
    'MAY MẶC': '#2CA02C',
    'GẠO': '#1F77B4'
}

# 5. Tạo biểu đồ 2 cột x 3 hàng
fig, axes = plt.subplots(
    3, 2,
    figsize=(11, 14),
    sharex=True,
    sharey=True
)

axes = axes.flatten()

# 6. Vẽ biểu đồ cho từng năm
for i, year in enumerate(years):

    ax = axes[i]

    data_year = data_plot[
        data_plot['Năm'] == year
    ]


    # Pivot dữ liệu
    pivot_data = data_year.pivot(
        index='Tháng',
        columns='Loại hàng',
        values='TEU'
    )

    # Bổ sung đủ 12 tháng và 5 nhóm hàng
    # Tháng không có bản ghi được tính bằng 0 TEU
    pivot_data = (
        pivot_data
        .reindex(
            index=range(1, 13),
            columns=mat_hang_giu
        )
        .fillna(0)
    )


    # Vẽ từng nhóm hàng
    for mat_hang in mat_hang_giu:

        ax.plot(
            pivot_data.index,
            pivot_data[mat_hang],
            marker='o',
            markersize=4,
            linewidth=1.8,
            color=colors[mat_hang],
            label=mat_hang
        )

    # Tiêu đề và trục

    ax.set_title(
        f'Năm {year}',
        fontsize=12,
        fontweight='bold',
        pad=8
    )

    # Hiển thị tên đầy đủ 12 tháng
    ax.set_xticks(range(1, 13))
    ax.set_xticklabels(
        [str(i) for i in range(1, 13)],
        fontsize=9
    )

    ax.tick_params(axis='x', labelbottom=True)

    ax.set_xlabel('Tháng', fontsize=10)
    ax.set_ylabel('Sản lượng (TEU)', fontsize=10)

    ax.grid(
        True,
        linestyle='--',
        alpha=0.35
    )

    ax.tick_params(axis='both', labelsize=9)

# 7. Ẩn ô biểu đồ thứ 6
axes[5].axis('off')

# 8. Tạo chú giải chung
handles, labels = axes[0].get_legend_handles_labels()

fig.legend(
    handles,
    labels,
    title='Nhóm hàng',
    loc='upper center',
    bbox_to_anchor=(0.5, 0.965),
    ncol=5,
    fontsize=9,
    title_fontsize=10,
    frameon=True
)

# 9. Tiêu đề chung
fig.suptitle(
    'Sản lượng TEU theo tháng của 5 nhóm hàng chủ lực (2021–2025)',
    fontsize=14,
    fontweight='bold',
    y=0.995
)

# 10. Điều chỉnh khoảng cách
plt.subplots_adjust(
    top=0.91,
    bottom=0.06,
    left=0.09,
    right=0.97,
    hspace=0.32,
    wspace=0.20
)

# 11. Lưu hình chất lượng cao
plt.savefig(
    'San_luong_TEU_5_nhom_hang_A4_doc.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()


#5. Tổng sản lượng container qua các năm

In [ ]:
# Lọc dữ liệu giai đoạn 2021–2025
df_plot = df[df['Năm'].isin([2021, 2022, 2023, 2024, 2025])].copy()

# Tính tổng TEU theo năm và tháng
monthly = (
    df_plot.groupby(['Năm', 'Tháng'])['TEU']
    .sum()
    .reset_index()
)

# Tạo cột thời gian từ Năm và Tháng
monthly['Thời gian'] = pd.to_datetime(
    monthly['Năm'].astype(str) + '-' +
    monthly['Tháng'].astype(str) + '-01'
)

# Sắp xếp theo thời gian
monthly = monthly.sort_values('Thời gian')

# Vẽ biểu đồ đường
plt.figure(figsize=(16, 6))

plt.plot(
    monthly['Thời gian'],
    monthly['TEU'],
    marker='o',
    linewidth=2
)

# Hiển thị nhãn trục X theo tháng/năm
plt.xticks(
    monthly['Thời gian'],
    monthly['Thời gian'].dt.strftime('%m/%Y'),
    rotation=45,
    ha='right'
)

plt.xlabel('Thời gian')
plt.ylabel('Sản lượng TEU')
plt.title('Tổng sản lượng TEU theo tháng giai đoạn 2021–2025')

plt.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()